# Gegen den trainierten Schach-Agenten spielen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mark-baumann/schach-reinforcement-lernen/blob/main/notebooks/02_play_against_agent.ipynb)

Dieses Notebook lädt ein mit [`01_train_chess_rl_agent.ipynb`](./01_train_chess_rl_agent.ipynb)
trainiertes Bewertungsnetz und lässt dich interaktiv per Texteingabe dagegen spielen — direkt im
Notebook, unabhängig von der Streamlit-App (`app.py`) im Hauptverzeichnis des Repos.

Das Notebook ist eigenständig lauffähig: Ist keine Checkpoint-Datei vorhanden, wird mit einem
**untrainierten** (zufällig initialisierten) Netz gespielt und eine Warnung ausgegeben — führe in
diesem Fall zuerst Notebook 1 aus oder lade deine eigene `chess_value_net.pt` hoch.


## 1. Setup

In [ ]:
%pip install -q python-chess


In [ ]:
import chess
import chess.svg
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import SVG, clear_output, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Modell-Definition

Muss identisch zur Definition in Notebook 1 sein, damit die gespeicherten Gewichte passen.

In [ ]:
PIECE_VALUES = {
    chess.PAWN: 1.0, chess.KNIGHT: 3.0, chess.BISHOP: 3.0,
    chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0,
}


def encode_board(board: chess.Board) -> torch.Tensor:
    planes = torch.zeros(12, 8, 8, dtype=torch.float32)
    for sq, piece in board.piece_map().items():
        row, col = divmod(sq, 8)
        idx = (piece.piece_type - 1) + (0 if piece.color == chess.WHITE else 6)
        planes[idx, row, col] = 1.0
    return planes


class ValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(12, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return torch.tanh(self.fc2(x)).squeeze(-1)


net = ValueNet().to(device)


@torch.no_grad()
def value_of(board: chess.Board) -> float:
    x = encode_board(board).unsqueeze(0).to(device)
    return net(x).item()


def select_ai_move(board: chess.Board) -> chess.Move:
    best_move, best_value = None, -float("inf")
    for move in board.legal_moves:
        board.push(move)
        value = -value_of(board)
        board.pop()
        if value > best_value:
            best_value, best_move = value, move
    return best_move


## 3. Checkpoint laden

Sucht `chess_value_net.pt` im aktuellen Verzeichnis (z. B. hochgeladen über den Dateibrowser
links in Colab, oder erzeugt von Notebook 1 im selben Colab-Runtime). Ohne gefundene Datei wird
mit einem untrainierten Netz weitergemacht.

In [ ]:
CHECKPOINT_PATH = "chess_value_net.pt"

try:
    net.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    net.eval()
    print(f"Checkpoint geladen: {CHECKPOINT_PATH}")
except FileNotFoundError:
    net.eval()
    print(
        f"Kein Checkpoint unter '{CHECKPOINT_PATH}' gefunden — es wird mit einem "
        "UNTRAINIERTEN Netz gespielt.\n"
        "Führe zuerst 01_train_chess_rl_agent.ipynb aus oder lade eine eigene "
        "chess_value_net.pt-Datei hoch (Dateibrowser links in Colab)."
    )


## 4. Interaktive Partie

Züge werden im UCI-Format eingegeben (z. B. `e2e4`, Umwandlung `e7e8q`). `quit` beendet die
Partie vorzeitig. Führe die Zelle erneut aus, um eine neue Partie zu starten.

In [ ]:
def render(board: chess.Board, human_color: bool):
    svg = chess.svg.board(board, flipped=(human_color == chess.BLACK), size=400)
    clear_output(wait=True)
    display(SVG(svg))


def play_interactive(human_color: bool = chess.WHITE, max_plies: int = 200):
    board = chess.Board()
    render(board, human_color)

    while not board.is_game_over() and board.fullmove_number * 2 <= max_plies:
        if board.turn == human_color:
            user_input = input("Dein Zug (UCI, z.B. e2e4) oder 'quit': ").strip().lower()
            if user_input in ("quit", "exit"):
                print("Partie abgebrochen.")
                return board
            try:
                move = chess.Move.from_uci(user_input)
            except ValueError:
                print("Ungültiges Format. Beispiel: e2e4")
                continue
            if move not in board.legal_moves:
                print("Ungültiger Zug — bitte erneut versuchen.")
                continue
            board.push(move)
        else:
            move = select_ai_move(board)
            board.push(move)

        render(board, human_color)

    if board.is_checkmate():
        winner = "Schwarz" if board.turn == chess.WHITE else "Weiß"
        print(f"Schachmatt! {winner} gewinnt.")
    elif board.is_game_over():
        print(f"Partie beendet: {board.result()}")
    else:
        print("Maximale Zugzahl erreicht.")
    return board


In [ ]:
# Zum Spielen ausführen. human_color=chess.BLACK, um als Schwarz zu spielen.
final_board = play_interactive(human_color=chess.WHITE)


## Hinweise

- Das Netz wählt seine Züge per 1-Ply-Greedy-Suche über das gelernte `ValueNet` — identisch zur
  Zugauswahl während des Self-Play-Trainings in Notebook 1.
- Für ein stärkeres Spiel: In Notebook 1 länger trainieren (`N_GAMES` erhöhen) und den
  resultierenden Checkpoint hier erneut laden.
- Die Streamlit-App (`app.py` im Projekt-Hauptverzeichnis) nutzt aktuell eine klassische
  Minimax-Suche mit handgeschriebener Bewertungsfunktion, keine gelernten Gewichte — die beiden
  Ansätze lassen sich gut vergleichen.
